In [1]:
import geopandas as gpd
import rasterio
from rasterio.features import rasterize
from rasterio.windows import Window
import numpy as np
import pandas as pd
from shapely.geometry import mapping

# -----------------------------
# Inputs
# -----------------------------
polygons_path = r"C:\Users\KyleSteen.AzureAD\Documents\NLCD\Python_Workspace\Alaska\Alaska_3_10.shp"
nlcd_path = r"C:\Users\KyleSteen.AzureAD\Documents\NLCD\Python_Workspace\Alaska\NLCD_Alaska_3338.tif"
out_csv = r"C:\Users\KyleSteen.AzureAD\Documents\NLCD\Python_Workspace\Alaska\NLCD_Alaska.csv"

# -----------------------------
# Logger
# -----------------------------
def log(msg):
    print(f"[{pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S')}] {msg}")

# -----------------------------
# Load polygons
# -----------------------------
log("Loading shapefile")
gdf = gpd.read_file(polygons_path)

if "ROW_ID" not in gdf.columns:
    gdf["ROW_ID"] = gdf.index
gdf.set_index("ROW_ID", inplace=True)

log("Fixing invalid geometries")
gdf["geometry"] = gdf["geometry"].buffer(0)

# -----------------------------
# Open NLCD raster
# -----------------------------
log("Opening NLCD raster")
src = rasterio.open(nlcd_path)
nodata_val = 255

# -----------------------------
# CRS info
# -----------------------------
log(f"Polygon CRS: {gdf.crs}")
log(f"Raster CRS: {src.crs}")

# -----------------------------
# NLCD classes mapping
# -----------------------------
nlcd_classes = {
    11: "Open Water",
    21: "Developed, Open Space",
    22: "Developed, Low Intensity",
    23: "Developed, Medium Intensity",
    24: "Developed, High Intensity",
    31: "Barren Land",
    41: "Deciduous Forest",
    42: "Evergreen Forest",
    43: "Mixed Forest",
    52: "Shrub/Scrub",
    71: "Grassland/Herbaceous",
    81: "Pasture/Hay",
    82: "Cultivated Crops",
    90: "Woody Wetlands",
    95: "Emergent Herbaceous Wetlands",
}

# -----------------------------
# Function to get majority NLCD value
# -----------------------------
def majority_nlcd(geom):
    minx, miny, maxx, maxy = geom.bounds

    # Create raster window
    window = src.window(minx, miny, maxx, maxy)

    # Safe window: ensure width/height >= 1
    width = max(int(np.ceil(window.width)), 1)
    height = max(int(np.ceil(window.height)), 1)
    safe_window = Window(window.col_off, window.row_off, width, height)

    # Read raster data
    data = src.read(1, window=safe_window)

    # Rasterize polygon
    mask = rasterize(
        [(mapping(geom), 1)],
        out_shape=data.shape,
        transform=src.window_transform(safe_window),
        fill=0,
        all_touched=True,
        dtype="uint8",
    )

    captured_pixels = np.count_nonzero(mask)
    if captured_pixels == 0:
        return None, None, 0

    values = data[mask == 1]
    values = values[values != nodata_val]
    if len(values) == 0:
        return None, None, captured_pixels

    counts = np.bincount(values)
    mode_val = np.argmax(counts)
    mode_class = nlcd_classes.get(mode_val, "Unknown")

    return mode_val, mode_class, captured_pixels

# -----------------------------
# Process polygons with progress
# -----------------------------
results = []
total_polys = len(gdf)
log(f"Processing {total_polys} polygons")

for i, (idx, row) in enumerate(gdf.iterrows(), start=1):
    geom = row.geometry
    code, cls, pixels = majority_nlcd(geom)

    results.append({
        "ROW_ID": idx,
        "NLCD_Code": code if code is not None else -1,
        "NLCD_Class": cls if cls is not None else "No Data",
        "Pixels_Captured": pixels,
    })

    # Print progress every 5% or last polygon
    if i % max(1, total_polys // 20) == 0 or i == total_polys:
        pct = (i / total_polys) * 100
        print(f"[{pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S')}] Progress: {pct:.1f}% ({i}/{total_polys})")

# -----------------------------
# Write CSV
# -----------------------------
df_out = pd.DataFrame(results)
df_out.to_csv(out_csv, index=False)
log(f"Writing CSV to {out_csv}")
log("Done")

[2026-03-10 11:51:46] Loading shapefile
[2026-03-10 11:51:46] Fixing invalid geometries
[2026-03-10 11:51:46] Opening NLCD raster
[2026-03-10 11:51:46] Polygon CRS: EPSG:3338
[2026-03-10 11:51:46] Raster CRS: EPSG:3338
[2026-03-10 11:51:46] Processing 2703 polygons
[2026-03-10 11:51:46] Progress: 5.0% (135/2703)
[2026-03-10 11:51:46] Progress: 10.0% (270/2703)
[2026-03-10 11:51:46] Progress: 15.0% (405/2703)
[2026-03-10 11:51:46] Progress: 20.0% (540/2703)
[2026-03-10 11:51:46] Progress: 25.0% (675/2703)
[2026-03-10 11:51:46] Progress: 30.0% (810/2703)
[2026-03-10 11:51:47] Progress: 35.0% (945/2703)
[2026-03-10 11:51:47] Progress: 40.0% (1080/2703)
[2026-03-10 11:51:47] Progress: 45.0% (1215/2703)
[2026-03-10 11:51:47] Progress: 49.9% (1350/2703)
[2026-03-10 11:51:47] Progress: 54.9% (1485/2703)
[2026-03-10 11:51:47] Progress: 59.9% (1620/2703)
[2026-03-10 11:51:47] Progress: 64.9% (1755/2703)
[2026-03-10 11:51:47] Progress: 69.9% (1890/2703)
[2026-03-10 11:51:47] Progress: 74.9% (202